# Pyro SAI — 00 Data discovery: find the daily G6-1.5K data and build the catalog

This notebook replaces `FindDaily.ipynb`. That notebook searched a placeholder `data_dir` (`/path/to/your/data/`), which is why every variable showed as "Missing".

1. **Inventory** the Reflective S3 buckets recursively. Every NetCDF file is classified by model, scenario, member, variable and frequency, using CAM names (`TREFHTMX`, `RHREFHT`, `U10`, `PRECT`), CMOR names (`tasmax`, `hurs`, `sfcWind`, `pr`) and stream/table tokens (`h1`, `day`, `Amon`, ...).
2. **Resolve** files of unknown frequency by opening one per group and measuring the time step.
3. *(Optional)* Add SSP2-4.5 **daily** baselines from the public Pangeo CMIP6 Zarr store (AWS) for models whose baseline is not on the Hub.
4. *(Optional)* Inspect the new **Earthmover/Arraylake Zarr stores** (CESM, MIROC) and register them.
5. **Write `pyrosai_catalog.json`**, which Tasks 1 and 2 read, and a readiness table.
6. **Write a small sample and a text report** to attach back to Claude for verification.

Nothing here is expensive: listing is metadata only, and the sample is a few MB.

In [ ]:
import os, sys, json, warnings
import numpy as np, pandas as pd, xarray as xr
sys.path.insert(0, os.getcwd())
import pyrosai as P
warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200, "display.width", 200)

## Configuration

In [ ]:
# Prefixes to search (add/remove as you learn where things are). Listing is recursive.
ROOTS = [
    "s3://reflective-persistent-prod-large/CESM2-WACCM/",
    "s3://reflective-persistent-prod-large/UKESM1-1/",
    "s3://reflective-persistent-prod-large/MIROC-ES2H/",
    "s3://reflective-persistent-prod-large/E3SMv3/",
    "s3://reflective-persistent-prod/alistairduffey/",
]
STORAGE_OPTIONS = None          # Hub credentials are picked up automatically
CATALOG_OUT = "pyrosai_catalog.json"
REPORT_OUT  = "pyrosai_discovery_report.txt"

if os.environ.get("PYROSAI_TEST") == "1":
    ROOTS = [os.environ["PYROSAI_SYNTH_ROOT"]]

## 1. Inventory

In [ ]:
listing = P.list_archive(ROOTS, storage_options=STORAGE_OPTIONS)
listing.to_csv("pyrosai_archive_listing.csv", index=False)
print(f"{len(listing)} NetCDF files listed")
display(listing.groupby(["model", "freq"]).size().unstack(fill_value=0))

## 2. Resolve unknown frequencies
One file is opened per model/scenario/variable group whose path gives no frequency hint.

In [ ]:
unk = listing[(listing.freq == "unknown") & listing["var"].notnull() & (listing.scenario != "?")]
for key, g in unk.groupby(["model", "scenario", "var"]):
    dt = P.probe_frequency(g.path.iloc[0], STORAGE_OPTIONS)
    f = "unknown" if not np.isfinite(dt) else ("day" if 0.9 <= dt <= 1.5 else
         ("subdaily" if dt < 0.9 else ("mon" if 27 <= dt <= 32 else f"{dt:.2f}d")))
    listing.loc[g.index, "freq"] = f
    print(key, "->", f, f"(dt={dt})")
print("daily files:", int((listing.freq == "day").sum()))

In [ ]:
day = listing[listing.freq == "day"]
cov = day.groupby(["model", "scenario", "member", "var"]).size().unstack("var", fill_value=0)
display(cov)
# Year coverage (from file names) of the FWI/VPD variables: need 2019-2039 (SSP2-4.5) and 2063-2083 (all)
fv = day[day["var"].isin(["tasmax", "tas", "hurs", "huss", "sfcWind", "uas", "pr"]) & (day.scenario != "?")]
yc = fv.groupby(["model", "scenario", "var"]).agg(first_year=("y0", "min"), last_year=("y1", "max"),
                                                    members=("member", "nunique"))
display(yc)

# Daily files whose variable is not one of ours: are there alternatives (e.g. min RH, gusts)?
other = day[day["var"].isnull()].copy()
other["token"] = other.path.str.split("/").str[-1].str.replace(r"[._-]\d{6,12}[-_]\d{6,12}.*$", "", regex=True)
print("Daily files with unrecognised variables (file-name stems, counts):")
display(other.groupby(["model", "token"]).size().sort_values(ascending=False).head(40))
print("Daily files with unknown scenario/member:")
display(day[(day.scenario == "?") | (day.member == "?")][["model", "scenario", "member", "var", "path"]].head(20))

## 3. *(Optional)* SSP2-4.5 daily baselines from Pangeo CMIP6 (AWS, public)
Use this only for models whose SSP2-4.5 **daily** output is not on the Hub. Also check that the model version matches: for example, the UKESM G6 runs use UKESM1-1 with its own SSP2-4.5 on the Hub, so the CMIP6 UKESM1-0-LL baseline would not be appropriate.

In [ ]:
ADD_PANGEO = {}    # e.g. {"CESM2-WACCM": "CESM2-WACCM"}  (Hub model name -> CMIP6 source_id)
pangeo_entries = {}
if ADD_PANGEO:
    cat_df = pd.read_csv("https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.csv")
    for hub_model, src in ADD_PANGEO.items():
        sub = cat_df.query("source_id == @src and experiment_id == 'ssp245' and table_id == 'day' "
                           "and variable_id in ['tasmax','tas','hurs','hursmin','sfcWind','pr','huss']")
        display(sub.groupby(["member_id", "variable_id"]).size().unstack(fill_value=0))
        for (mem, var), g in sub.groupby(["member_id", "variable_id"]):
            pangeo_entries.setdefault(hub_model, {}).setdefault(mem, {})[var] = {
                "kind": "zarr", "store": g.zstore.iloc[-1], "storage_options": {"anon": True}, "var": var}

## 4. *(Optional)* Earthmover / Arraylake Zarr stores (CESM, MIROC G6-1.5K)
Copy the few lines that open the store from the Hub example (`/shared/Code_example/Earthmover_zarr_GeoMIP/`) into `open_earthmover_root()` below, then run the cell. It prints the group tree so you can see whether **daily** fields exist. If they do, fill `EARTHMOVER_ENTRIES`. Entries of `kind: "earthmover"` go through the same function, so Tasks 1 and 2 need no other change. Paste the printed tree into the report if you are unsure.

In [ ]:
USE_EARTHMOVER = False

def open_earthmover_root(entry=None):
    # >>> paste from /shared/Code_example/Earthmover_zarr_GeoMIP, e.g. (API may differ):
    # from arraylake import Client
    # repo = Client().get_repo("<org>/<repo-name>")
    # session = repo.readonly_session("main")
    # return session.store
    raise NotImplementedError("paste the store-opening lines from the Hub example here")

def open_earthmover(entry):
    store = open_earthmover_root(entry)
    return xr.open_zarr(store, group=entry.get("group"), consolidated=False, chunks={})

P.CUSTOM_OPENERS["earthmover"] = open_earthmover

# Fill after inspecting the tree:  {model: {scenario: {member: {canonical_var: {"kind": "earthmover", "group": "...", "var": "..."}}}}}
EARTHMOVER_ENTRIES = {}

if USE_EARTHMOVER:
    import zarr
    root = zarr.open_group(open_earthmover_root(), mode="r")
    print(root.tree())

## 5. Build and save the catalog

In [ ]:
catalog = P.build_catalog(listing)
for hub_model, mems in pangeo_entries.items():
    catalog.setdefault(hub_model, {}).setdefault("ssp245", {})
    for mem, vars_ in mems.items():
        catalog[hub_model]["ssp245"].setdefault(mem, {}).update(vars_)
for model, scens in EARTHMOVER_ENTRIES.items():
    for scen, mems in scens.items():
        for mem, vars_ in mems.items():
            catalog.setdefault(model, {}).setdefault(scen, {}).setdefault(mem, {}).update(vars_)
P.save_catalog(catalog, CATALOG_OUT)
inv = P.catalog_inventory(catalog)
display(inv)

need1, need2 = ["tasmax", "hurs", "sfcWind", "pr"], ["tas", "hurs", "pr"]
ws = P.Workspace(catalog, ".")
rt = P.readiness_table(catalog, ["tasmax", "tas", "hurs", "sfcWind", "pr"])
display(rt.pivot_table(index=["model", "member"], columns="scenario", values="missing", aggfunc="first").fillna(""))
ready1 = [m for m in catalog if all(ws.members(m, s, [P.choose_fwi_temp(ws, m), "hurs", "sfcWind", "pr"]) for s in P.SCENARIOS)]
print("Ready for Task 1 (FWI):", {m: P.choose_fwi_temp(ws, m) for m in ready1})
print("Ready for Task 2 (VPD):", ws.models(need2))
for m in catalog:
    for s in P.SCENARIOS:
        if s not in catalog[m]:
            print(f"  [missing] {m}: no daily {s}")

## 6. Report and small sample for Claude
This writes `pyrosai_discovery_report.txt` (a text summary plus the raw `xarray` description of one file per variable) and `pyrosai_sample_<model>.nc`. The sample covers one member, a 20°×30° box, and 2 years of each run, and is about 10 MB. **Attach both files** to the chat.

In [ ]:
SAMPLE_MODEL = None                    # None = first model ready for Task 1
SAMPLE_BBOX  = (-125, -95, 30, 50)     # lon0, lon1, lat0, lat1 (western North America)
SAMPLE_YEARS = {"ssp245": [(2029, 2030), (2073, 2074)], "G6-1.5K-SAI": [(2073, 2074)], "G6-1.5K-HiLLA": [(2073, 2074)]}

lines = ["PYRO SAI DISCOVERY REPORT", "=" * 80, f"roots: {ROOTS}", "",
         "files per model/frequency:", listing.groupby(["model", "freq"]).size().unstack(fill_value=0).to_string(), "",
         "daily coverage (files):", cov.to_string() if len(day) else "(no daily files found)", "",
         "catalog inventory:", inv.to_string(), ""]
for m, scens in catalog.items():
    for s, mems in scens.items():
        mem = sorted(mems)[0]
        for v, e in mems[mem].items():
            lines.append(f"--- {m} / {s} / {mem} / {v}  ({len(e.get('paths', []))} files) e.g. {e.get('paths', [e.get('store')])[0]}")
            try:
                ds = P.open_entry(e, P.ALIASES.get(v, []))
                lines.append(str(ds))
                if "time" in ds:
                    lines.append(f"    time[0:3]={ds.time.values[:3]}  calendar={ds.time.dt.calendar}")
            except Exception as ex:
                lines.append(f"    OPEN FAILED: {type(ex).__name__}: {ex}")
open(REPORT_OUT, "w").write("\n".join(lines))
print(f"wrote {REPORT_OUT} ({os.path.getsize(REPORT_OUT)/1e3:.0f} kB)")

ready = ready1
if ready:
    m = SAMPLE_MODEL or ready[0]
    t = P.choose_fwi_temp(ws, m)
    P.prepare_wind_corrections(P.Workspace(catalog, ".", bbox=SAMPLE_BBOX), [m], "./pyrosai_output/metrics")
    parts = []
    for s, spans in SAMPLE_YEARS.items():
        mem = ws.members(m, s, [t, "hurs", "sfcWind", "pr"])[0]
        for yrs in spans:
            d = P.load_inputs(catalog, m, s, mem, ["tasmax", "hursmin", "hurs", "sfcWind", "pr", "tas"], yrs, SAMPLE_BBOX)
            d = d.expand_dims(run=[f"{s}_{mem}_{yrs[0]}-{yrs[1]}"])
            d = d.assign_coords(time=np.arange(d.sizes["time"])).rename(time="day_index")
            parts.append(d.load())
    sample = xr.concat(parts, "run", join="outer")
    fn = f"pyrosai_sample_{m}.nc"
    sample.to_netcdf(fn)
    print(f"wrote {fn} ({os.path.getsize(fn)/1e6:.1f} MB)")
else:
    print("No model is ready for Task 1 yet - attach the report so the parsing rules can be adjusted.")